# 🛡️ CyberSec AI 10-Weeks Interactive Colab Notebook
## An Ninh Mạng & Ứng Dụng AI Thế Hệ Mới (Dành Cho Mobile & Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

Notebook này cho phép học viên thực thi trực tiếp toàn bộ các bài thực hành Python từ Tuần 1 đến Tuần 10 (Socket, Scapy, Nmap Audit, Bcrypt Hashing, OSINT Prompting, Web Log Audit, và AI Anomaly Detection) trên môi trường điện toán đám mây Google Colab từ máy tính hoặc điện thoại.

### 📦 Bước 1: Cài đặt công cụ Nmap & các thư viện Python

In [ ]:
# Cài đặt nmap và các thư viện cần thiết trong Colab
!apt-get update -qq && !apt-get install -y -qq nmap > /dev/null
!pip install -q scapy bcrypt pandas scikit-learn requests

### 🔌 Tuần 1: Lập Trình Python Socket Communication

In [ ]:
import socket

# Kiểm tra kết nối TCP Localhost
sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
sock.settimeout(1.0)
result = sock.connect_ex(('127.0.0.1', 80))
print(f"[+] Port 80 Check Result: {'OPEN' if result == 0 else 'CLOSED'}")
sock.close()

### 🔎 Tuần 2: Fast Port Scanner đa luồng

In [ ]:
import concurrent.futures

COMMON_PORTS = [21, 22, 80, 443, 8080]

def scan_port(port):
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.settimeout(0.5)
    res = s.connect_ex(('127.0.0.1', port))
    s.close()
    return port, (res == 0)

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    results = executor.map(scan_port, COMMON_PORTS)
    for p, open_status in results:
        print(f"[+] Port {p:5d} : {'OPEN' if open_status else 'CLOSED'}")

### 🔍 Tuần 5: Quét Nmap Tự Động Trên Localhost

In [ ]:
import subprocess

res = subprocess.run(["nmap", "-sV", "--open", "-p", "1-100", "127.0.0.1"], capture_output=True, text=True)
print(res.stdout)

### 🌐 Tuần 6: Phân Tích Gói Tin Scapy

In [ ]:
from scapy.all import IP, TCP, Ether

# Sinh gói tin giả lập để kiểm tra cấu trúc
pkt = Ether()/IP(dst="127.0.0.1")/TCP(dport=80, flags="S")
print(f"[+] Simulated Packet Summary: {pkt.summary()}")
print(f"[+] IP Source: {pkt[IP].src} -> IP Dst: {pkt[IP].dst}")

### 🔐 Tuần 7: Mã Hóa Bcrypt Mật Khẩu

In [ ]:
import bcrypt

pw = "UserSecretPassword@2026"
salt = bcrypt.gensalt(rounds=10)
hashed = bcrypt.hashpw(pw.encode('utf-8'), salt)

print(f"[+] Bcrypt Salted Hash: {hashed.decode('utf-8')}")
print(f"[+] Verify Result     : {bcrypt.checkpw(pw.encode('utf-8'), hashed)}")

### 🤖 Tuần 8: Tự Động Trích Xuất OSINT Threat Intel

In [ ]:
import re, json

sample_report = "Threat IP 192.168.1.100 connected to C2 45.33.32.156 with SHA256 e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855."
ips = re.findall(r'\b(?:[0-9]{1,3}\.){3}[0-9]{1,3}\b', sample_report)
hashes = re.findall(r'\b[A-Fa-f0-9]{64}\b', sample_report)

print(json.dumps({"extracted_ips": ips, "extracted_hashes": hashes}, indent=2))

### 📝 Tuần 9: Parse Web Access Log Phát Hiện Tấn Công

In [ ]:
logs = [
    '127.0.0.1 - - [27/Jul/2026] "GET /index.html HTTP/1.1" 200 1024',
    '127.0.0.1 - - [27/Jul/2026] "GET /login?user=admin\' OR 1=1-- HTTP/1.1" 200 512'
]

sqli_regex = re.compile(r"(?i)(\'|OR%201=1|UNION|SELECT)")
for l in logs:
    if sqli_regex.search(l):
        print(f"[⚠️ ALERT - SQL INJECTION]: {l}")
    else:
        print(f"[✅ BENIGN]: {l}")

### 🤖 Tuần 10: AI SOC Monitoring (Isolation Forest Anomaly Detection)

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import IsolationForest

# Synthetic data generation
normal_reqs = np.random.normal(75, 10, 95)
anomaly_reqs = np.random.normal(500, 50, 5)
reqs = np.concatenate([normal_reqs, anomaly_reqs])

df = pd.DataFrame({'requests_per_min': reqs})
model = IsolationForest(contamination=0.05, random_state=42)
df['anomaly'] = model.fit_predict(df[['requests_per_min']])

anomalies = df[df['anomaly'] == -1]
print(f"[+] Total records: {len(df)}")
print(f"[⚠️ ALERT] Anomalies Detected by Isolation Forest: {len(anomalies)}")
print(anomalies.head())